In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install dionysus==2.0.10
import dionysus as d
import networkx as nx # Network structures
import numpy as np # Numpy arrays and operations
import random # Random sampling for network models
from itertools import combinations, product # For getting different simplices and all combinations of lists

import matplotlib.pyplot as plt # Plotting
import time # Timing simulations
from tqdm.notebook import trange, tqdm # Allows for real-time progress bar of simulations

import gc # Memory management
import pickle # Takes environment variables and saves them as is
import gzip # Allows for compression of saved files
from joblib import Parallel, delayed # Parallelization functions
import multiprocessing # Get number of cpu cores
import os # Helpful functions, mainly for checking if a file exists

import collections # Used for obtaining degree distribution
import math
import sys

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for dionysus: filename=dionysus-2.0.10-cp312-cp312-linux_x86_64.whl size=437482 sha256=7bb264908df9d92bb4b9d26710bbc06ac93a7c6c7b2ab98ba57191c98295cdd7
  Stored in directory: /root/.cache/pip/wheels/11/9f/ca/ae26580795e766a8b6dce75ae4f29163b83b52d7a72efd5d48
Successfully built dionysus


In [ ]:
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

from GenerativeHypergraphModels import *

In [ ]:
n = 40; k = 4; p = 0.25
H, Betti, SimplexCounts, Euler = HG_ErdosRenyi_kUnif(n, k, p, timing = True)
Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
Upper_sum = np.cumsum(Upper)
Lower_sum = np.cumsum(Lower)

(1/5) Initializing hypergraph, variables and data structures
Initialization complete, time taken : 0.05624794960021973 seconds
(2/5) Beginning network evolution


  0%|          | 0/91390 [00:00<?, ?it/s]

Evolution complete, time taken : 0.30330491065979004 seconds
(3/5) Beginning Persistent Homology
Persistent Homology complete, time taken : 334.57731533050537 seconds
(4/5) Beginning Betti number extraction
Betti number extraction complete, time taken : 0.15429449081420898 seconds
(5/5) Beginning Euler Characteristic extraction
Euler Characteristic extraction complete, time taken : 0.22115254402160645 seconds


100%|██████████| 22848/22848 [02:56<00:00, 129.54it/s]


In [ ]:
def PC_HG_ErdosRenyi(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; k = params[1]; p = params[2]; iteration = params[3]; model = params[4]

  # Create filename from params
  if model == 'kunif':
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/ErdosRenyi/kunif_ER_'+str(n)+'_'+str(k)+'_'+str(p).replace('.','_')+'_'+str(iteration)+'.pkl'
    if os.path.isfile(filename):
      return 0
    H, Betti, SimplexCounts, Euler = HG_ErdosRenyi_kUnif(n, k, p, timing = False)
    Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
    Upper_sum = np.cumsum(Upper)
    Lower_sum = np.cumsum(Lower)
    data = [Betti, SimplexCounts, Euler, Upper_sum, Lower_sum, SF, FES]
  elif model == 'regular':
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/ErdosRenyi/ER_'+str(n)+'_'+str(k)+'_'+str(p).replace('.','_')+'_'+str(iteration)+'.pkl'
    if os.path.isfile(filename):
      return 0
    H, Betti, SimplexCounts, EdgeCounts, Euler = HG_ErdosRenyi(n, k, p, timing = False)
    Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
    Upper_sum = np.cumsum(Upper)
    Lower_sum = np.cumsum(Lower)
    data = [Betti, SimplexCounts, EdgeCounts, Euler, Upper_sum, Lower_sum, SF, FES]
  else:
    print("Invalid model")
    return 0


  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0

#N = [40]; K = [3,4,5]; P = [0.25]; iterations = list(range(1,11)); models = ['kunif']
#N = [40]; K = [3,4,5]; P = [0.1]; iterations = list(range(1,11)); models = ['regular']
N = [5,10,15,20,25,30,35]; K = [3,4,5]; P = [1]; iterations = list(range(1,11)); models = ['kunif','regular']
params = [N, K, P, iterations, models]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(PC_HG_ErdosRenyi)(param) for param in tqdm(params))

  0%|          | 0/420 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
def PC_HG_BarabasiAlbert(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; k = params[1]; p = params[2]; iteration = params[3];

  # Create filename from params
  if p == 0:
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/BarabasiAlbert/BA_'+str(n)+'_'+str(k)+'_'+str(iteration)+'.pkl'
  else:
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/BarabasiAlbert/Simplicial_BA_'+str(n)+'_'+str(k)+'_'+str(p).replace('.','_')+'_'+str(iteration)+'.pkl'
  if os.path.isfile(filename):
    return 0

  H, D, Betti, SimplexCounts, Euler = HG_PreferentialAttachment_Simplicial(k, p, n, timing = False)
  Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
  Upper_sum = np.cumsum(Upper)
  Lower_sum = np.cumsum(Lower)
  data = [D, Betti, SimplexCounts, Euler, Upper_sum, Lower_sum, SF, FES]

  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0

N = [10000]; K = [3,4,5,6,7,8,9,10,15]; P = [0]; iterations = [1,2,3,4,5,6,7,8,9,10];
#N = [10000]; K = [3,4,5,6,7,8]; P = np.linspace(0,1,21); iterations = [1,2,3,4,5,6,7,8,9,10];
params = [N, K, P, iterations]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(PC_HG_BarabasiAlbert)(param) for param in tqdm(params))

  0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def PC_HG_NL_BarabasiAlbert(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; k = params[1]; alpha = params[2]; iteration = params[3];
  print(n)
  # Create filename from params
  if alpha == 1:
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/BarabasiAlbert/BA_'+str(n)+'_'+str(k)+'_'+str(iteration)+'.pkl'
  elif alpha == 0:
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/BarabasiAlbert/RA_'+str(n)+'_'+str(k)+'_'+str(iteration)+'.pkl'
  else:
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/BarabasiAlbert/Nonlinear_BA_'+str(n)+'_'+str(k)+'_'+str(alpha).replace('.','_')+'_'+str(iteration)+'.pkl'
  if os.path.isfile(filename):
    return 0

  H, D, Betti, SimplexCounts, Euler = HG_NL_PreferentialAttachment_kUnif(k, n, alpha, timing = False)
  data = [D, Betti, SimplexCounts, Euler]

  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0


N = [10000]; K = [3,4,5,6,7,8,9,10]; alphas = np.linspace(0,2,41); iterations = [1,2,3,4,5,6,7,8,9,10];
params = [N, K, alphas, iterations]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(PC_HG_NL_BarabasiAlbert)(param) for param in tqdm(params))

In [ ]:
n = 10000; k = 10; alpha = 1.2
H, D, Betti, SimplexCounts, Euler = HG_NL_PreferentialAttachment_kUnif(k, n, alpha, timing = True)

(1/5) Initializing hypergraph, variables and data structures
Initialization complete, time taken : 0.004797935485839844 seconds
(2/5) Beginning network evolution


  0%|          | 0/9991 [00:00<?, ?it/s]

Evolution complete, time taken : 63.68937683105469 seconds
(3/5) Beginning Persistent Homology
Persistent Homology complete, time taken : 4.791473388671875 seconds
(4/5) Beginning Betti number extraction
Betti number extraction complete, time taken : 5.717129230499268 seconds
(5/5) Beginning Euler Characteristic extraction
Euler Characteristic extraction complete, time taken : 0.32264232635498047 seconds


In [ ]:
def HG_WattsStrogatz_kUnif(n, K, timing = False):
    """
    ...
    Input: int n, the number of nodes in H
           int K, the maximum allowable edge size
           bool timing, whether to display timing information
    Output: Betti, an array of the 0, 1 and 2 Betti numbers for each time step
            SimplexCounts
    """
    if timing:
        print("(1/5) Initializing hypergraph, variables and data structures",flush=True)
        start = time.time()

    nodes = range(n)

    # Generate a list of all edges of size <= K that can be formed from n node ring lattice
    E = [tuple(sorted([(v+i) % n for i in range(K)])) for v in nodes]
    Edges = set()
    Edges.update(E)
    H = {i: e for i, e in enumerate(E)}
    # By shuffing this list, it is equivalent to forming a filtration where
    # edges are rewired u.a.r
    random.shuffle(E)

    # Use set structure to remember what simplices have been added
    Simplices = set(tuple([v]) for v in nodes)
    # Intialize simplex counts with n vertices
    # The maximum simplex dimension is either n-1 or maxDim
    SimplexCounts = np.zeros( (len(E)+1 , min(K, n)) )
    SimplexCounts[0][0] = n

    # Initialize dictionary which keeps track of when simplices
    # are added and removed
    Times = {tuple([v]) : [0] for v in nodes}
    timer = 0
    SuperSets = dict()

    # Need to keep track of how many supersets there are for the initial
    # simplices, as these are the only simplices that can be removed, which
    # only happens when 1. they are not an edge and 2. are not contained within
    # any other edges.
    for e in E:
        Simplices.add(e)
        SimplexCounts[0][len(e)-1] += 1
        Times[e] = [0]
        for size in range(2, len(e)):
            for subset in combinations(e, size):
                if subset not in SuperSets:
                    SuperSets[subset] = 1
                    Simplices.add(subset)
                    SimplexCounts[0][len(subset)-1] += 1
                    Times[subset] = [0]
                else:
                    SuperSets[subset] += 1

    if timing:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(2/5) Beginning network evolution",flush=True)
        start = time.time()

    for e in tqdm(E) if timing else E:
        timer += 1; SimplexCounts[timer] = SimplexCounts[timer-1].copy();
        # e is being rewired so we remove it from edges
        H[-(timer + len(E) - 1)] = e
        Edges.remove(e)
        Simplices.remove(e)
        SimplexCounts[timer][len(e)-1] -= 1
        Times[e].append(timer)

        # Since e is rewired, we have to update its subsets
        for size in range(2, len(e)):
            # For each subset, decrease the number of supersets by 1
            for subset in combinations(e, size):
                SuperSets[subset] -= 1
                # If the subset has no supersets and is not itself an edge,
                # rewiring the edge destroys the simplex
                if SuperSets[subset] == 0 and subset not in Edges:
                    Simplices.remove(subset)
                    SimplexCounts[timer][len(subset)-1] -= 1
                    Times[subset].append(timer)

        # Rewire edge
        source = np.random.choice(e)
        while True:
            newTargets = set([source])
            # Randomly sample new targets
            while len(newTargets) < len(e):
                newTarget = np.random.choice(nodes)
                newTargets.add(newTarget)
            newEdge = tuple(sorted(newTargets))
            if newEdge not in Edges:
                Edges.add(newEdge)
                H[timer + len(E) - 1] = newEdge
                break

        # Update new edge
        if newEdge not in Simplices:
            Simplices.add(newEdge)
            SimplexCounts[timer][len(newEdge)-1] += 1
            if newEdge not in Times:
                Times[newEdge] = [timer]
            else:
                Times[newEdge].append(timer)

        # We have a new edge so we have to update data structures for subsets
        for size in range(2, len(newEdge)):
            # For each subset, decrease the number of supersets by 1
            for subset in combinations(newEdge, size):
                # If the subset was not already a simplex, it is now
                if subset not in Simplices:
                    Simplices.add(subset)
                    SimplexCounts[timer][len(subset)-1] += 1
                    if subset not in Times:
                        Times[subset] = [timer]
                    else:
                        Times[subset].append(timer)
                # if the subset is a simplex AND is one of our initial edges, increment supersets
                elif subset in SuperSets:
                    SuperSets[subset] += 1

    if timing:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
        start = time.time()

    # Extract list of every simplex added/removed, and list of times they were
    # added/removed, for input into zigzag persistence.
    simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

    # Clear out Times, which is massive
    del(Times); del(Simplices); del(SuperSets); del(Edges); gc.collect()

    # Construct filtration and compute homology
    f = d.Filtration(simplices)
    zz, dgms, cells = d.zigzag_homology_persistence(f, times)

    # Clear out remaining lists which are massive
    del(simplices); del(times); gc.collect()

    if timing:
        end = time.time()
        print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(4/5) Beginning Betti number extraction",flush=True)
        start = time.time()

    # PH doesn't return betti numbers, it returns persistence pairs
    # Here we loop through pairs, any time between the birth
    # and death of the pair corresponds to the existence of a hole
    Betti = np.zeros((4,timer+1))
    one = np.ones(timer+1)
    for i, dgm in enumerate(dgms):
        for p in dgm:
            Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

    if timing:
        end = time.time()
        print("Betti number extraction complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(5/5) Beginning Euler Characteristic extraction",flush=True)
        start = time.time()

    # For each time step, compute the euler characteristic as the alternating
    # sum of simplex counts SUM( (-1)^j * num j-simplices )
    Euler = np.zeros(timer+1)
    for i in range(len(SimplexCounts)):
        for j in range(len(SimplexCounts[i])):
            Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

    if timing:
        end = time.time()
        print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds",flush=True)

    return H, Betti, SimplexCounts, Euler

In [ ]:
def HG_WattsStrogatz(n, K, timing = False):
    """
    ...
    Input: int n, the number of nodes in H
           int K, the maximum allowable edge size
           bool timing, whether to display timing information
    Output: Betti, an array of the 0, 1 and 2 Betti numbers for each time step
            SimplexCounts
    """
    if timing:
        print("(1/5) Initializing hypergraph, variables and data structures",flush=True)
        start = time.time()

    nodes = range(n)

    # Generate a list of all edges of size <= K that can be formed from n nodes
    E = [tuple(sorted([(v+i) % n for i in range(size)])) for v in nodes for size in range(2, K+1)]
    Edges = set()
    Edges.update(E)
    H = {i: e for i, e in enumerate(E)}
    # By shuffing this list, it is equivalent to forming a filtration where
    # edges are rewired u.a.r
    random.shuffle(E)

    # Use set structure to remember what simplices have been added
    Simplices = set(tuple([v]) for v in nodes)
    # Intialize simplex counts with n vertices
    # The maximum simplex dimension is either n-1 or maxDim
    SimplexCounts = np.zeros( (len(E)+1 , min(K, n)) )
    SimplexCounts[0][0] = n

    # Initialize dictionary which keeps track of when simplices
    # are added and removed
    Times = {tuple([v]) : [0] for v in nodes}
    timer = 0
    SuperSets = dict()

    # Need to keep track of how many supersets there are for the initial
    # simplices, as these are the only simplices that can be removed, which
    # only happens when 1. they are not an edge and 2. are not contained within
    # any other edges.
    for e in E:
        if e not in SuperSets:
            SuperSets[e] = 0
            Simplices.add(e)
            SimplexCounts[0][len(e)-1] += 1
            Times[e] = [0]
        for size in range(2, len(e)):
            for subset in combinations(e, size):
                if subset not in SuperSets:
                    SuperSets[subset] = 1
                    Simplices.add(subset)
                    SimplexCounts[0][len(subset)-1] += 1
                    Times[subset] = [0]
                else:
                    SuperSets[subset] += 1

    if timing:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(2/5) Beginning network evolution",flush=True)
        start = time.time()

    for e in tqdm(E) if timing else E:
        timer += 1; SimplexCounts[timer] = SimplexCounts[timer-1].copy();
        # e is being rewired so we remove it from edges
        H[-(timer + len(E) - 1)] = e
        Edges.remove(e)
        # If the removed edge has no supersets, then the simplex is destroyed
        if (e not in SuperSets) or (SuperSets[e] == 0):
            Simplices.remove(e)
            SimplexCounts[timer][len(e)-1] -= 1
            Times[e].append(timer)

        # Since e is rewired, we have to update its subsets
        for size in range(2, len(e)):
            # For each subset, decrease the number of supersets by 1
            for subset in combinations(e, size):
                SuperSets[subset] -= 1
                # If the subset has no supersets and is not itself an edge,
                # rewiring the edge destroys the simplex
                if SuperSets[subset] == 0 and subset not in Edges:
                    Simplices.remove(subset)
                    SimplexCounts[timer][len(subset)-1] -= 1
                    Times[subset].append(timer)

        # Rewire edge
        source = np.random.choice(e)
        while True:
            newTargets = set([source])
            # Randomly sample new targets
            while len(newTargets) < len(e):
                newTarget = np.random.choice(nodes)
                newTargets.add(newTarget)
            newEdge = tuple(sorted(newTargets))
            if newEdge not in Edges:
                Edges.add(newEdge)
                H[timer + len(E) - 1] = newEdge
                break

        # Update new edge
        if newEdge not in Simplices:
            Simplices.add(newEdge)
            SimplexCounts[timer][len(newEdge)-1] += 1
            if newEdge not in Times:
                Times[newEdge] = [timer]
            else:
                Times[newEdge].append(timer)

        # We have a new edge so we have to update data structures for subsets
        for size in range(2, len(newEdge)):
            # For each subset, decrease the number of supersets by 1
            for subset in combinations(newEdge, size):
                # If the subset was not already a simplex, it is now
                if subset not in Simplices:
                    Simplices.add(subset)
                    SimplexCounts[timer][len(subset)-1] += 1
                    if subset not in Times:
                        Times[subset] = [timer]
                    else:
                        Times[subset].append(timer)
                # if the subset is a simplex AND is one of our initial edges, increment supersets
                elif subset in SuperSets:
                    SuperSets[subset] += 1

    if timing:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
        start = time.time()

    # Extract list of every simplex added/removed, and list of times they were
    # added/removed, for input into zigzag persistence.
    simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

    # Clear out Times, which is massive
    del(Times); del(Simplices); del(SuperSets); del(Edges); gc.collect()

    # Construct filtration and compute homology
    f = d.Filtration(simplices)
    zz, dgms, cells = d.zigzag_homology_persistence(f, times)

    # Clear out remaining lists which are massive
    del(simplices); del(times); gc.collect()

    if timing:
        end = time.time()
        print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(4/5) Beginning Betti number extraction",flush=True)
        start = time.time()

    # PH doesn't return betti numbers, it returns persistence pairs
    # Here we loop through pairs, any time between the birth
    # and death of the pair corresponds to the existence of a hole
    Betti = np.zeros((4,timer+1))
    one = np.ones(timer+1)
    for i, dgm in enumerate(dgms):
        for p in dgm:
            Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

    if timing:
        end = time.time()
        print("Betti number extraction complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(5/5) Beginning Euler Characteristic extraction",flush=True)
        start = time.time()

    # For each time step, compute the euler characteristic as the alternating
    # sum of simplex counts SUM( (-1)^j * num j-simplices )
    Euler = np.zeros(timer+1)
    for i in range(len(SimplexCounts)):
        for j in range(len(SimplexCounts[i])):
            Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

    if timing:
        end = time.time()
        print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds",flush=True)

    return H, Betti, SimplexCounts, Euler

In [ ]:
def PC_HG_WattsStrogatz(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; k = params[1]; iteration = params[2]; model = params[3]

  # Create filename from params
  if model == 'kunif':
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/WattsStrogatz/kunif_WS_'+str(n)+'_'+str(k)+'_'+str(iteration)+'.pkl'
    if os.path.isfile(filename):
      return 0
    H, Betti, SimplexCounts, Euler = HG_WattsStrogatz_kUnif(n, k, timing = False)
    data = [Betti, SimplexCounts, Euler]
  elif model == 'regular':
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/WattsStrogatz/WS_'+str(n)+'_'+str(k)+'_'+str(iteration)+'.pkl'
    if os.path.isfile(filename):
      return 0
    H, Betti, SimplexCounts, Euler = HG_WattsStrogatz(n, k, timing = False)
    Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
    Upper_sum = np.cumsum(Upper)
    Lower_sum = np.cumsum(Lower)
    data = [Betti, SimplexCounts, Euler, Upper_sum, Lower_sum, SF, FES]
  else:
    print("Invalid model")
    return 0


  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0


N = [1000]; K = [3,4,5,6,7]; iterations = list(range(1,11)); models = ['kunif']
params = [N, K, iterations, models]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(PC_HG_WattsStrogatz)(param) for param in tqdm(params))

  0%|          | 0/100 [00:00<?, ?it/s]

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGSEGV(-11)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
n = 1000; k = 8
H, Betti, SimplexCounts, Euler = HG_WattsStrogatz_kUnif(n, k, timing = True)
Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
Upper_sum = np.cumsum(Upper)
Lower_sum = np.cumsum(Lower)

(1/5) Initializing hypergraph, variables and data structures
Initialization complete, time taken : 0.774165153503418 seconds
(2/5) Beginning network evolution


  0%|          | 0/1000 [00:00<?, ?it/s]

Evolution complete, time taken : 2.1078901290893555 seconds
(3/5) Beginning Zigzag Persistent Homology
Zigzag Persistent Homology complete, time taken : 61.56406831741333 seconds
(4/5) Beginning Betti number extraction
Betti number extraction complete, time taken : 0.030167579650878906 seconds
(5/5) Beginning Euler Characteristic extraction
Euler Characteristic extraction complete, time taken : 0.01772284507751465 seconds


100%|██████████| 2000/2000 [00:00<00:00, 2978.77it/s]


In [ ]:
def HG_WattsStrogatz_Simplicial(n, K, p, timing = False):
    """
    ...
    Input: int n, the number of nodes in H
           int K, the maximum allowable edge size
           bool timing, whether to display timing information
    Output: Betti, an array of the 0, 1 and 2 Betti numbers for each time step
            SimplexCounts
    """
    if timing:
        print("(1/5) Initializing hypergraph, variables and data structures",flush=True)
        start = time.time()

    nodes = range(n)

    # Generate a list of all edges of size <= K that can be formed from n nodes
    E = [tuple(sorted([(v+i) % n for i in range(K)])) for v in nodes]
    for v in nodes:
        for size in range(2,K):
            for comb in combinations(range(v+1,v+K),size-1):
                if random.random() < p:
                    E.append(tuple(sorted([v]+[u % n for u in comb])))

    Edges = set()
    Edges.update(E)
    H = {i: e for i, e in enumerate(E)}
    # By shuffing this list, it is equivalent to forming a filtration where
    # edges are rewired u.a.r
    random.shuffle(E)

    # Use set structure to remember what simplices have been added
    Simplices = set(tuple([v]) for v in nodes)
    # Intialize simplex counts with n vertices
    # The maximum simplex dimension is either n-1 or maxDim
    SimplexCounts = np.zeros( (len(E)+1 , min(K, n)) )
    SimplexCounts[0][0] = n

    # Initialize dictionary which keeps track of when simplices
    # are added and removed
    Times = {tuple([v]) : [0] for v in nodes}
    timer = 0
    SuperSets = dict()

    # Need to keep track of how many supersets there are for the initial
    # simplices, as these are the only simplices that can be removed, which
    # only happens when 1. they are not an edge and 2. are not contained within
    # any other edges.
    for e in E:
        if e not in SuperSets:
            SuperSets[e] = 0
            Simplices.add(e)
            SimplexCounts[0][len(e)-1] += 1
            if len(e) <= 4:
                Times[e] = [0]
        for size in range(2, len(e)):
            for subset in combinations(e, size):
                if subset not in SuperSets:
                    SuperSets[subset] = 1
                    Simplices.add(subset)
                    SimplexCounts[0][len(subset)-1] += 1
                    if len(subset) <= 4:
                        Times[subset] = [0]
                else:
                    SuperSets[subset] += 1

    if timing:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(2/5) Beginning network evolution",flush=True)
        start = time.time()

    for e in tqdm(E) if timing else E:
        timer += 1; SimplexCounts[timer] = SimplexCounts[timer-1].copy();
        # e is being rewired so we remove it from edges
        H[-(timer + len(E) - 1)] = e
        Edges.remove(e)
        # If the removed edge has no supersets, then the simplex is destroyed
        if (e not in SuperSets) or (SuperSets[e] == 0):
            Simplices.remove(e)
            SimplexCounts[timer][len(e)-1] -= 1
            if len(e) <= 4:
                Times[e].append(timer)

        # Since e is rewired, we have to update its subsets
        for size in range(2, len(e)):
            # For each subset, decrease the number of supersets by 1
            for subset in combinations(e, size):
                SuperSets[subset] -= 1
                # If the subset has no supersets and is not itself an edge,
                # rewiring the edge destroys the simplex
                if SuperSets[subset] == 0 and subset not in Edges:
                    Simplices.remove(subset)
                    SimplexCounts[timer][len(subset)-1] -= 1
                    if len(subset) <= 4:
                        Times[subset].append(timer)

        # Rewire edge
        source = np.random.choice(e)
        while True:
            newTargets = set([source])
            # Randomly sample new targets
            while len(newTargets) < len(e):
                newTarget = np.random.choice(nodes)
                newTargets.add(newTarget)
            newEdge = tuple(sorted(newTargets))
            if newEdge not in Edges:
                Edges.add(newEdge)
                H[timer + len(E) - 1] = newEdge
                break

        # Update new edge
        if newEdge not in Simplices:
            Simplices.add(newEdge)
            SimplexCounts[timer][len(newEdge)-1] += 1
            if len(newEdge) <= 4:
                if newEdge not in Times:
                    Times[newEdge] = [timer]
                else:
                    Times[newEdge].append(timer)

        # We have a new edge so we have to update data structures for subsets
        for size in range(2, len(newEdge)):
            # For each subset, decrease the number of supersets by 1
            for subset in combinations(newEdge, size):
                # If the subset was not already a simplex, it is now
                if subset not in Simplices:
                    Simplices.add(subset)
                    SimplexCounts[timer][len(subset)-1] += 1
                    if len(subset) <= 4:
                        if subset not in Times:
                            Times[subset] = [timer]
                        else:
                            Times[subset].append(timer)
                # if the subset is a simplex AND is one of our initial edges, increment supersets
                elif subset in SuperSets:
                    SuperSets[subset] += 1

    if timing:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
        start = time.time()

    # Extract list of every simplex added/removed, and list of times they were
    # added/removed, for input into zigzag persistence.
    simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

    # Clear out Times, which is massive
    del(Times); del(Simplices); del(SuperSets); del(Edges); gc.collect()

    # Construct filtration and compute homology
    f = d.Filtration(simplices)
    zz, dgms, cells = d.zigzag_homology_persistence(f, times)

    # Clear out remaining lists which are massive
    del(simplices); del(times); gc.collect()

    if timing:
        end = time.time()
        print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(4/5) Beginning Betti number extraction",flush=True)
        start = time.time()

    # PH doesn't return betti numbers, it returns persistence pairs
    # Here we loop through pairs, any time between the birth
    # and death of the pair corresponds to the existence of a hole
    Betti = np.zeros((4,timer+1))
    one = np.ones(timer+1)
    for i, dgm in enumerate(dgms):
        for p in dgm:
            Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

    if timing:
        end = time.time()
        print("Betti number extraction complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(5/5) Beginning Euler Characteristic extraction",flush=True)
        start = time.time()

    # For each time step, compute the euler characteristic as the alternating
    # sum of simplex counts SUM( (-1)^j * num j-simplices )
    Euler = np.zeros(timer+1)
    for i in range(len(SimplexCounts)):
        for j in range(len(SimplexCounts[i])):
            Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

    if timing:
        end = time.time()
        print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds",flush=True)

    return H, Betti, SimplexCounts, Euler

In [ ]:
def PC_HG_Simplicial_WattsStrogatz(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; k = params[1]; p = params[2]; iteration = params[3];

  # Create filename from params
  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/WattsStrogatz/WS_Simplicial_'+str(n)+'_'+str(k)+'_'+str(p).replace('.','_')+'_'+str(iteration)+'.pkl'
  if os.path.isfile(filename):
      return 0
  H, Betti, SimplexCounts, Euler = HG_WattsStrogatz_Simplicial(n, k, p, timing = False)
  Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)
  Upper_sum = np.cumsum(Upper)
  Lower_sum = np.cumsum(Lower)
  data = [Betti, SimplexCounts, Euler, Upper_sum, Lower_sum, SF[len(H)//3:], FES[len(H)//3:]]

  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0


N = [500]; K = [3,4,5,6]; P = np.linspace(0,1,21); iterations = list(range(1,101));
params = [N, K, P, iterations]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = 2#multiprocessing.cpu_count()

# Actual call to parallel function
for param in tqdm(params):
    results = PC_HG_Simplicial_WattsStrogatz(param)
#results = Parallel(n_jobs=num_cores)(delayed(PC_HG_Simplicial_WattsStrogatz)(param) for param in tqdm(params))

NameError: name 'np' is not defined

In [ ]:
n = 500; k = 7; p = 1
H, Betti, SimplexCounts, Euler = HG_WattsStrogatz_Simplicial(n, k, p, timing = True)
Upper, Lower, SF, FES = simpliciality_process_hypergraph(H)

(1/5) Initializing hypergraph, variables and data structures
Initialization complete, time taken : 0.35547947883605957 seconds
(2/5) Beginning network evolution


  0%|          | 0/31500 [00:00<?, ?it/s]

Evolution complete, time taken : 10.172849655151367 seconds
(3/5) Beginning Zigzag Persistent Homology


In [ ]:
def PC_PA_HypergraphModel(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  m = params[0]; steps = params[1]; iteration = params[2]
  # Create filename from params
  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_HypergraphModel/PA_HypergraphModel_'+str(m)+'_'+str(steps)+'_'+str(iteration)+'.pkl'
  # Check if current file already exists, if it does exist we do not
  # want to waste time running it again
  if os.path.isfile(filename):
    return 0
  data = PA_HypergraphModel(m, steps, timing = True)
  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0

# Set N to be list containing numbers of nodes to be in the network; P to be the
# terminal edge density
M = [2,3,4,5,6,7,8,9,10,12,14,16,18,20,30,40,50,100]; STEPS = [10000000]; iteration = [1,2,3,4,5,6,7,8,9,10]
# Given lists of parameter values, get every pair of parameter values
# I.E take the crossproduct of the sets of parameters
params = [M, STEPS, iteration]
params = [p for p in product(*params)]

# Get number of cpus that can be used
#num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
#results = PC_PA_HypergraphModel(params[0])
#results = Parallel(n_jobs=num_cores)(delayed(PC_PA_HypergraphModel)(param) for param in tqdm(params))

for param in tqdm(params):
    results = PC_PA_HypergraphModel(param)

  0%|          | 0/180 [00:00<?, ?it/s]

NameError: name 'PA_HypergraphModel' is not defined

In [ ]:
def BarabasiAlbert(n,k,timing = False):
  """
  Input: n - The total number of nodes in the terminal network
         k - The number of edges between to a newly added node and
             existing nodes in the network chosen preferentially
  Output: A random scale-free Barabasi-Albert graph with n nodes
          as a networkx object.
  """
  # Initialize BA graph as star graph with k+1 vertices, node 0 connects to nodes 1, 2, ... , k
  G = nx.star_graph(k)
  V = list(range(n))

  # Each time an edge is connected to a node, add a copy of that
  # node to this list. Then randomly sampling from this list
  # is equivalent to preferential attachment.
  repeated_nodes = ([0] * k) + [i for i in range(1,k+1)]

  for v in (tqdm(V[k+1:]) if timing else V[k+1:]):
    # Select k targets in graph using preferential attachment
    targets = set()
    while len(targets) < k:
        x = random.choice(repeated_nodes)
        targets.add(x)
    targets = list(targets)
    repeated_nodes.extend(targets + [v] * k)

    # Add node v and update its degree
    G.add_node(v);

    # Add new edges and update target degrees
    for target in targets:
      G.add_edge(target, v);

  return G

def PC_BarabasiAlbert(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; k = params[1]; iteration = params[2]
  # Create filename from params
  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/BarabasiAlbert/BarabasiAlbert_'+str(n)+'_'+str(k)+'_'+str(iteration)+'.pkl'
  # Check if current file already exists, if it does exist we do not
  # want to waste time running it again
  if os.path.isfile(filename):
    return 0
  data = BarabasiAlbert(n, k, timing = True)
  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0

# Set N to be list containing numbers of nodes to be in the network; P to be the
# terminal edge density
N = [10000000]; K = [1,2,3,4,5,6,7,8,9,10]; iteration = [1,2,3,4,5,6,7,8,9,10]
# Given lists of parameter values, get every pair of parameter values
# I.E take the crossproduct of the sets of parameters
params = [N, K, iteration]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
#results = PC_BarabasiAlbert(params[0])
#results = Parallel(n_jobs=num_cores)(delayed(PC_PA_HypergraphModel)(param) for param in tqdm(params))

for param in tqdm(params):
    results = PC_BarabasiAlbert(param)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/9999998 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def PA_Tree_HypergraphModel(m, steps, timing=False):
  """
  Input: m - integer number of nodes to add at each timestep.
             Model adds hyperedges of size m+1.
         steps - integer number of iterations for which to run the model.
             Hypergraph ends with (m * steps) + 1 nodes and (steps) hyperedges.
  Output: Hypergraph dictionary H of hypernetwork from "Evolving Hypernetwork Model"
          along with the degree frequency distribution D.
  """
  # Initialize node list V, and hypergraph as a dictionary
  # Start with 1 node, add m nodes at each step, so at the end
  # there will be m * steps + 1 many nodes in total
  V = list(range(m * steps + 1))
  H = {}

  # Keep track of vertex degrees for degree distribution
  D = np.array([1 if i == 0 else 0 for i in range(m * steps + 1)])
  total_deg = 1
  # Each time an edge is connected to a node, add a copy of that
  # node to this list. Then randomly sampling from this list
  # is equivalent to preferential attachment.
  repeated_nodes = [0]

  for step in (tqdm(range(steps)) if timing else range(steps)):
    # Select target in graph using preferential attachment
    # At the current step, the first step * m + 1 nodes have been added so far
    # So these are the nodes we can preferentially attach to
    target = random.choice(repeated_nodes)
    # Add the m new nodes to the new hyperedge
    H_edge = [target] + [step*m + 1 + i for i in range(m)]
    H[step] = H_edge
    repeated_nodes.extend(H_edge)
    # Update node degrees
    for v in H_edge: D[v] += 1

  # Returns D as a dictionary with degree values as keys and
  # the number of nodes with the given degree as entries.
  D =  collections.Counter(D)

  return H, D

def PC_PA_Tree_HypergraphModel(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  m = params[0]; steps = params[1]; iteration = params[2]
  # Create filename from params
  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_Tree_HypergraphModel/PA_Tree_HypergraphModel_'+str(m)+'_'+str(steps)+'_'+str(iteration)+'.pkl'
  # Check if current file already exists, if it does exist we do not
  # want to waste time running it again
  if os.path.isfile(filename):
    return 0
  data = PA_Tree_HypergraphModel(m, steps, timing = True)
  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0

# Set N to be list containing numbers of nodes to be in the network; P to be the
# terminal edge density
#M = [1]; STEPS = [1000000]; iteration = [1]
# Given lists of parameter values, get every pair of parameter values
# I.E take the crossproduct of the sets of parameters
#params = [M, STEPS, iteration]
#params = [p for p in product(*params)]

params = [(2,5000000,1),(3,3333333,1),(4,2500000,1),(5,2000000,1),(6,1666667,1),(7,1428571,1),(8,1250000,1),(9,1111111,1),(10,1000000,1),
          (2,5000000,2),(3,3333333,2),(4,2500000,2),(5,2000000,2),(6,1666667,2),(7,1428571,2),(8,1250000,2),(9,1111111,2),(10,1000000,2),
          (2,5000000,3),(3,3333333,3),(4,2500000,3),(5,2000000,3),(6,1666667,3),(7,1428571,3),(8,1250000,3),(9,1111111,3),(10,1000000,3),
          (2,5000000,4),(3,3333333,4),(4,2500000,4),(5,2000000,4),(6,1666667,4),(7,1428571,4),(8,1250000,4),(9,1111111,4),(10,1000000,4),
          (2,5000000,5),(3,3333333,5),(4,2500000,5),(5,2000000,5),(6,1666667,5),(7,1428571,5),(8,1250000,5),(9,1111111,5),(10,1000000,5),
          (2,5000000,6),(3,3333333,6),(4,2500000,6),(5,2000000,6),(6,1666667,6),(7,1428571,6),(8,1250000,6),(9,1111111,6),(10,1000000,6),
          (2,5000000,7),(3,3333333,7),(4,2500000,7),(5,2000000,7),(6,1666667,7),(7,1428571,7),(8,1250000,7),(9,1111111,7),(10,1000000,7),
          (2,5000000,8),(3,3333333,8),(4,2500000,8),(5,2000000,8),(6,1666667,8),(7,1428571,8),(8,1250000,8),(9,1111111,8),(10,1000000,8),
          (2,5000000,9),(3,3333333,9),(4,2500000,9),(5,2000000,9),(6,1666667,9),(7,1428571,9),(8,1250000,9),(9,1111111,9),(10,1000000,9),
          (2,5000000,10),(3,3333333,10),(4,2500000,10),(5,2000000,10),(6,1666667,10),(7,1428571,10),(8,1250000,10),(9,1111111,10),(10,1000000,10)]


# Get number of cpus that can be used
#num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
#results = PC_PA_Tree_HypergraphModel(params[0])
#results = Parallel(n_jobs=num_cores)(delayed(PC_PA_Tree_HypergraphModel)(param) for param in tqdm(params))

for param in tqdm(params):
    results = PC_PA_Tree_HypergraphModel(param)

In [ ]:
def PA_Poisson_Binomial_HypergraphModel(lam, p, steps, timing=False): # P_x = 'default',
  """
  Input: P_y - A list/array object describing the probability distribution for the
               sampled hyperedge sizes Y_t
         P_x - A list/array object describing the probability distribution for the
               sampled number of nodes to add at each timestep X_t
         steps - integer number of iterations for which to run the model.
  Output: Hypergraph dictionary H along with the degree frequency distribution D.
  """

  # Initialize node list V, and hypergraph as a dictionary will lam many
  # nodes belonging to a hyperedge of size lam
  V = list(range(lam))
  H = {0:[i for i in range(lam)]}

  # Keep track of vertex degrees for degree distribution
  D = [1] * lam
  # We will use repeated_nodes as a fast, but storage intensive way to
  # preferentially draw nodes. Each node v with appear deg(v) many times.
  repeated_nodes = [i for i in range(lam)];

  for step in (tqdm(range(1,steps+1)) if timing else range(1,steps+1)):
    # Select the size of Yt from the Poisson distribution
    Y_t = np.random.poisson(lam)
    # Select the number of nodes Xt from a binomial distribution with
    # E[Xt | Yt] = p*Yt
    X_t = np.random.binomial(Y_t, p)
    targets = set()

    # Check to make sure there are enough nodes to wire to
    # If not, add more new nodes instead of pref wiring
    if (Y_t - X_t) > len(V):
        X_t += (Y_t - X_t) - len(V)
    # Select (Y_t - X_t) many targets in graph using preferential attachment
    while len(targets) < (Y_t - X_t):
        x = random.choice(repeated_nodes)
        targets.add(x)
    targets = list(targets)
    # Add the X_t many new nodes to the new hyperedge
    new_nodes = [V[-1] + 1 + i for i in range(X_t)]
    V.extend(new_nodes); D.extend([0] * X_t)
    H_edge = targets + new_nodes; H[step] = H_edge
    repeated_nodes.extend(H_edge)

    # Update node degrees
    for v in H_edge: D[v] += 1

  # Returns D as a dictionary with degree values as keys and
  # the number of nodes with the given degree as entries.
  D =  collections.Counter(D)

  return H, D

def PC_PA_Poisson_Binomial_HypergraphModel(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  lam = params[0]; p = params[1]; N = params[2]; iteration = params[3]
  # By setting the number of steps in this way, we ensure that the
  # EXPECTED number of nodes at termination is at least N + N0.
  steps = math.ceil(N / (p * lam))
  # Create filename from params
  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_Poisson_Binomial_HypergraphModel/PA_Poisson_Binomial_HypergraphModel_'+str(lam)+'_'+str(p).replace('.','_')+'_'+str(N)+'_'+str(iteration)+'.pkl'
  # Check if current file already exists, if it does exist we do not
  # want to waste time running it again
  if os.path.isfile(filename):
    return 0
  data = PA_Poisson_Binomial_HypergraphModel(lam, p, steps, timing = True)
  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0

# Set N to be list containing numbers of nodes to be in the network; P to be the
# terminal edge density
lam = [5]; p = [.01,.1,.2,.3,.4,.5]; N = [100000]; iteration = [1,2,3,4,5] #lam = [5,10,15,20,30,40,50,100]; p = [.01,.05,.1,.2,.3,.4,.5,.6,.7,.8,.9]; N = [1000000]; iteration = [1,2,3,4,5,6,7,8,9,10]
# Given lists of parameter values, get every pair of parameter values
# I.E take the crossproduct of the sets of parameters
params = [lam, p, N, iteration]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
#results = PC_PA_UniformEdgeDist_HypergraphModel(params[0])
results = Parallel(n_jobs=num_cores)(delayed(PC_PA_Poisson_Binomial_HypergraphModel)(param) for param in tqdm(params))

#for p in tqdm(P):
 # for iteration in tqdm(iterations):
#for param in tqdm(params):
#   results = PC_PA_Poisson_Binomial_HypergraphModel(param)

  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
def PA_Dist_From_Data_HypergraphModel(Xt, Yt, timing=False):
  """
  Input:
         steps - integer number of iterations for which to run the model.
  Output: Hypergraph dictionary H along with the degree frequency distribution D.
  """

  # Initialize node list V, and hypergraph as a dictionary will max(Yt) many
  # nodes belonging to a hyperedge of size max(Yt)
  V = list(range(Yt[0]))
  H = {0:[i for i in range(Yt[0])]}

  # Keep track of vertex degrees for degree distribution
  D = [1] * Yt[0]
  # We will use repeated_nodes as a fast, but storage intensive way to
  # preferentially draw nodes. Each node v with appear deg(v) many times.
  repeated_nodes = [i for i in range(Yt[0])];

  for step in (tqdm(range(1,len(Yt))) if timing else range(1,len(Yt))):
    ##random_int = np.random.randint(0,len(Yt))
    # Select the size of Yt
    ##yt = Yt[random_int]
    yt = Yt[step]
    # Select the number of nodes Xt
    xt = Xt[step]
    ##xt = Xt[random_int]
    targets = set()

    # Select (Y_t - X_t) many targets in graph using preferential attachment
    while len(targets) < (yt - xt):
        x = random.choice(repeated_nodes)
        targets.add(x)
    targets = list(targets)
    # Add the X_t many new nodes to the new hyperedge
    new_nodes = [V[-1] + 1 + i for i in range(xt)]
    V.extend(new_nodes); D.extend([0] * xt)
    H_edge = targets + new_nodes; H[step] = H_edge
    repeated_nodes.extend(H_edge)

    # Update node degrees
    for v in H_edge: D[v] += 1

  # Returns D as a dictionary with degree values as keys and
  # the number of nodes with the given degree as entries.
  D =  collections.Counter(D)

  return H, D

In [ ]:
import numpy as np
import random
import collections
from tqdm import tqdm
import bisect


class WeightedSampler:
    """Efficient data structure for weighted sampling with dynamic updates"""

    def __init__(self, initial_weights):
        self.weights = list(initial_weights)
        self.cumulative = self._build_cumulative()
        self.total_weight = self.cumulative[-1] if self.cumulative else 0.0

    def _build_cumulative(self):
        """Build cumulative sum array for binary search"""
        cumulative = []
        total = 0.0
        for w in self.weights:
            total += w
            cumulative.append(total)
        return cumulative

    def update_weight(self, node_id, new_weight):
        """Update weight for a specific node"""
        if node_id >= len(self.weights):
            # Extend arrays for new nodes
            while len(self.weights) <= node_id:
                self.weights.append(0.0)
                self.cumulative.append(self.cumulative[-1] if self.cumulative else 0.0)

        old_weight = self.weights[node_id]
        weight_diff = new_weight - old_weight

        self.weights[node_id] = new_weight

        # Update cumulative sums from this point forward
        for i in range(node_id, len(self.cumulative)):
            self.cumulative[i] += weight_diff

        self.total_weight = self.cumulative[-1] if self.cumulative else 0.0

    def sample(self, exclude=None):
        """Sample a node using binary search on cumulative weights"""
        if exclude is None:
            exclude = set()

        if self.total_weight <= 0:
            return None

        # For small exclude sets, use rejection sampling (more efficient)
#        if len(exclude) < 10:
#            max_attempts = min(50, len(self.weights))
#            for _ in range(max_attempts):
        while True:
            target = np.random.random() * self.total_weight
            idx = bisect.bisect_left(self.cumulative, target)
            if idx < len(self.weights) and idx not in exclude and self.weights[idx] > 0:
                return idx

            # Fallback: direct sampling from valid nodes
#            valid_nodes = [i for i, w in enumerate(self.weights)
#                          if i not in exclude and w > 0]
#            if valid_nodes:
#                weights_subset = [self.weights[i] for i in valid_nodes]
#                total_subset = sum(weights_subset)
#                if total_subset > 0:
#                    probs = [w/total_subset for w in weights_subset]
#                    return np.random.choice(valid_nodes, p=probs)
#        else:
            # For large exclude sets, use direct sampling
#            valid_nodes = [i for i, w in enumerate(self.weights)
#                          if i not in exclude and w > 0]
#            if valid_nodes:
#                weights_subset = [self.weights[i] for i in valid_nodes]
#                total_subset = sum(weights_subset)
#                if total_subset > 0:
#                    probs = [w/total_subset for w in weights_subset]
#                    return np.random.choice(valid_nodes, p=probs)

        return None


def Nonlinear_PA_Dist_From_Data_HypergraphModel(Xt, Yt, alpha=1.0, timing=False):
    """
    Input:
         Xt - list of number of new nodes to add at each step
         Yt - list of hyperedge sizes at each step
         alpha - nonlinear preferential attachment parameter
         timing - boolean to show progress bar
    Output: Hypergraph dictionary H along with the degree frequency distribution D.
    """

    num_nodes = Yt[0]
    H = {0: [i for i in range(Yt[0])]}

    # Keep track of vertex degrees for degree distribution
    D = [1] * Yt[0]

    # Initialize efficient weighted sampler
    initial_weights = [1.0] * Yt[0]
    sampler = WeightedSampler(initial_weights)

    total_nodes = sum(Xt)
    D.extend([0] * (total_nodes - Yt[0]))

    for step in (tqdm(range(1, len(Yt))) if timing else range(1, len(Yt))):
        # Select the size of Yt and number of nodes Xt
        yt = Yt[step]
        xt = Xt[step]

        targets_needed = yt - xt
        targets = set()

        while len(targets) < targets_needed:
            x = sampler.sample(exclude=targets)
            if x is not None:
                targets.add(x)
            else:
                break

        targets = list(targets)

        # Add the X_t many new nodes to the new hyperedge
        new_nodes = [num_nodes + i for i in range(xt)]
        num_nodes += xt
        H_edge = targets + new_nodes
        H[step] = H_edge

        # Batch update degrees and weights
        for v in H_edge:
            D[v] += 1
            new_degree = D[v]
            new_weight = new_degree ** alpha
            sampler.update_weight(v, new_weight)

    D = collections.Counter(D)

    return H, D


# Example usage with different alpha values:
# alpha = 1.0  # Linear preferential attachment (original behavior)
# alpha > 1.0  # Super-linear (rich get richer effect amplified)
# alpha < 1.0  # Sub-linear (more egalitarian attachment)
# alpha = 0.0  # Uniform random attachment

In [ ]:
name = 'email-Enron'
loaddata = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/Processed/XtYt_FromData/'+str(name)+'.pkl'
with gzip.open(loaddata, 'rb') as f:
  extracted_info = pickle.load(f)
extracted_info.reverse()
Xt = []; Yt = []
for entry in extracted_info:
  Yt.append(entry['edge_size'])
  Xt.append(entry['isolated_nodes'])

In [ ]:
names = ['email-Enron']#['coauth-dblp','coauth-MAG-history', 'coauth-MAG-geology','congress-bills','email-Enron','email-EU','threads-ask-ubuntu','threads-math-sx',
         #'contact-high-school','contact-primary-school','hospital-lyon','malawi-village',
         #'science-gallery','SFHH-conference','hypertext-conference','InVS13','InVS15',
         #'tags-ask-ubuntu','tags-math-sx']
         #'coauth-dblp', 'coauth-MAG-history', 'coauth-MAG-geology', 'threads-stack-overflow', 'tags-stack-overflow'

steps = list(range(1,51))

for name in tqdm(names):
  loaddata = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/Processed/XtYt_FromData/'+str(name)+'.pkl'
  with gzip.open(loaddata, 'rb') as f:
    extracted_info = pickle.load(f)
  # Xt Yt's are recorded in reverse order, so we reverse them back
  extracted_info.reverse()
  Xt = []; Yt = []
  for entry in extracted_info:
    Yt.append(entry['edge_size'])
    Xt.append(entry['isolated_nodes'])

  for step in tqdm(steps):
    # Check if current file already exists, if it does exist we do not
    # want to waste time running it again
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/'+str(name)+'_'+str(step)+'.pkl'
    if os.path.exists(filename):
      print(f"File {filename} already exists. Skipping...")
      continue

    data = PA_Dist_From_Data_HypergraphModel(Xt, Yt)
    # Pickle and save the output
    with gzip.open(filename,'wb') as f:
      pickle.dump(data, f);
    print(f"Saved {filename}")

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

NameError: name 'PA_Dist_From_Data_HypergraphModel' is not defined

In [ ]:
names = ['hospital-lyon']
          #['email-Enron','email-EU',
         #'contact-high-school','contact-primary-school','hospital-lyon','malawi-village',
         #'science-gallery','SFHH-conference','hypertext-conference','InVS13','InVS15']
         #'tags-ask-ubuntu','tags-math-sx']
         #'congress-bills','coauth-dblp','coauth-MAG-history', 'coauth-MAG-geology','threads-ask-ubuntu','threads-math-sx'

steps = list(range(1,101))
alphas = np.linspace(0.1,10,100) #np.linspace(0.1,2,20)

for name in tqdm(names):
  loaddata = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/Processed/XtYt_FromData/'+str(name)+'.pkl'
  with gzip.open(loaddata, 'rb') as f:
    extracted_info = pickle.load(f)
  # Xt and Yt's are recorded in reverse order so we reverse them back
  extracted_info.reverse()
  Xt = []; Yt = []
  for entry in extracted_info:
    Yt.append(entry['edge_size'])
    Xt.append(entry['isolated_nodes'])

  for alpha in tqdm(alphas):
    for step in tqdm(steps):
      # Check if current file already exists, if it does exist we do not
      # want to waste time running it again
      filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/'+str(name)+'/Nonlinear_'+f"{alpha:.1f}".replace('.','_')+'_'+str(name)+'_'+str(step)+'.pkl'
      if os.path.exists(filename):
        print(f"File {filename} already exists. Skipping...")
        continue

      data = Nonlinear_PA_Dist_From_Data_HypergraphModel(Xt, Yt, alpha, timing=False)
      # Pickle and save the output
      with gzip.open(filename,'wb') as f:
        pickle.dump(data, f);
      print(f"Saved {filename}")

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  1%|          | 1/100 [00:07<12:17,  7.45s/it]

File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_1.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_2.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_3.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_4.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_5.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_6.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_7.pkl alread



 21%|██        | 21/100 [00:08<00:22,  3.53it/s]

Saved /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_21.pkl
Saved /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_22.pkl




 23%|██▎       | 23/100 [00:09<00:24,  3.15it/s]

...

Saved /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_38.pkl




 39%|███▉      | 39/100 [00:18<00:35,  1.70it/s]

Saved /content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/hospital-lyon/Nonlinear_0_1_hospital-lyon_39.pkl


  0%|          | 0/1 [00:22<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
alphas = np.linspace(0.2,3,15)
alphas

array([0.2, 0.4, 0.6, 0.8, 1. , 1.2, 1.4, 1.6, 1.8, 2. , 2.2, 2.4, 2.6,
       2.8, 3. ])

In [ ]:
def PC_Nonlinear_PA(params):
  name, alpha, step = params

  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/PA_From_Data/'+str(name)+'/Nonlinear_'+f"{alpha:.1f}".replace('.','_')+'_'+str(name)+'_'+str(step)+'.pkl'
  if os.path.exists(filename):
    print(f"File {filename} already exists. Skipping...")
    return 1

  loaddata = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/Processed/XtYt_FromData/'+str(name)+'.pkl'
  with gzip.open(loaddata, 'rb') as f:
    extracted_info = pickle.load(f)
  # Xt and Yt's are recorded in reverse order so we reverse them back
  extracted_info.reverse()
  Xt = []; Yt = []
  for entry in extracted_info:
    Yt.append(entry['edge_size'])
    Xt.append(entry['isolated_nodes'])

  data = Nonlinear_PA_Dist_From_Data_HypergraphModel(Xt, Yt, alpha, timing=False)
  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  print(f"Saved {filename}")

  return 0

#names = ['hospital-lyon','email-EU',
#         'contact-high-school','contact-primary-school','hospital-lyon','malawi-village',
#         'science-gallery','SFHH-conference','hypertext-conference','InVS13','InVS15',
#         'tags-ask-ubuntu','tags-math-sx',
names =  ['coauth-MAG-history', 'coauth-MAG-geology','threads-ask-ubuntu','threads-math-sx'] #'congress-bills',
steps = range(1,6); alphas = np.linspace(0.2,3,15)
#np.linspace(0.1,10,100)
#params = [(name,alphas,steps) for name in names]

params = [names, alphas, steps]
params = [p for p in product(*params)]

num_cores = 8
results = Parallel(n_jobs=num_cores)(delayed(PC_Nonlinear_PA)(param) for param in tqdm(params))

  5%|▌         | 16/300 [00:20<00:28, 10.02it/s]

In [ ]:
def RA_Dist_From_Data_HypergraphModel(Xt, Yt, timing=False):
  """
  Input:
         steps - integer number of iterations for which to run the model.
  Output: Hypergraph dictionary H along with the degree frequency distribution D.
  """

  # Initialize node list V, and hypergraph as a dictionary
  V = list(range(Yt[0]))
  H = {0:[i for i in range(Yt[0])]}

  # Keep track of vertex degrees for degree distribution
  D = [1] * Yt[0]

  for step in (tqdm(range(1,len(Yt))) if timing else range(1,len(Yt))):
    # Select the size of Yt
    yt = Yt[step]
    # Select the number of nodes Xt
    xt = Xt[step]
    targets = set()

    # Select (Y_t - X_t) many targets in graph using preferential attachment
    while len(targets) < (yt - xt):
        x = random.choice(V)
        targets.add(x)
    targets = list(targets)
    # Add the X_t many new nodes to the new hyperedge
    new_nodes = [V[-1] + 1 + i for i in range(xt)]
    V.extend(new_nodes); D.extend([0] * xt)
    H_edge = targets + new_nodes; H[step] = H_edge

    # Update node degrees
    for v in H_edge: D[v] += 1

  # Returns D as a dictionary with degree values as keys and
  # the number of nodes with the given degree as entries.
  D =  collections.Counter(D)

  return H, D

In [ ]:
names = ['hospital-lyon','coauth-MAG-history', 'coauth-MAG-geology','email-Enron','email-EU','threads-ask-ubuntu','threads-math-sx',
         'contact-high-school','contact-primary-school','hospital-lyon','malawi-village',
         'science-gallery','SFHH-conference','hypertext-conference','InVS13','InVS15',
         'tags-ask-ubuntu','tags-math-sx']
         #'coauth-dblp']


steps = list(range(1,101))

for name in tqdm(names):
  loaddata = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/Processed/XtYt_FromData/'+str(name)+'.pkl'
  with gzip.open(loaddata, 'rb') as f:
    extracted_info = pickle.load(f)
  extracted_info.reverse()
  Xt = []; Yt = []
  for entry in extracted_info:
    Yt.append(entry['edge_size'])
    Xt.append(entry['isolated_nodes'])

  for step in tqdm(steps):
    # Check if current file already exists, if it does exist we do not
    # want to waste time running it again
    filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/'+str(name)+'/RA_'+str(name)+'_'+str(step)+'.pkl'
    if os.path.exists(filename):
      print(f"File {filename} already exists. Skipping...")
      continue

    data = RA_Dist_From_Data_HypergraphModel(Xt, Yt)
    # Pickle and save the output
    with gzip.open(filename,'wb') as f:
      pickle.dump(data, f);
    print(f"Saved {filename}")

100%|██████████| 100/100 [00:00<00:00, 5602.19it/s]


File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_1.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_2.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_3.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_4.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_5.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_6.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/hospital-lyon/RA_hospital-lyon_7.pkl already exists. Skipping...
File /content/drive/My Drive/Colab Notebooks/Hypergraph


  1%|          | 1/100 [00:02<04:12,  2.55s/it]

Saved /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/coauth-MAG-history/RA_coauth-MAG-history_1.pkl



  2%|▏         | 2/100 [00:04<03:57,  2.42s/it]

Saved /content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/coauth-MAG-history/RA_coauth-MAG-history_2.pkl


  6%|▌         | 1/18 [00:08<02:17,  8.09s/it]


KeyboardInterrupt: 

In [ ]:
def PC_RA(params):
  name, step = params

  filename = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/RA_From_Data/'+str(name)+'/RA_'+str(name)+'_'+str(step)+'.pkl'
  if os.path.exists(filename):
    print(f"File {filename} already exists. Skipping...")
    return 1

  loaddata = '/content/drive/My Drive/Colab Notebooks/Hypergraphs/Processed/XtYt_FromData/'+str(name)+'.pkl'
  with gzip.open(loaddata, 'rb') as f:
    extracted_info = pickle.load(f)
  # Xt and Yt's are recorded in reverse order so we reverse them back
  extracted_info.reverse()
  Xt = []; Yt = []
  for entry in extracted_info:
    Yt.append(entry['edge_size'])
    Xt.append(entry['isolated_nodes'])

  data = RA_Dist_From_Data_HypergraphModel(Xt, Yt, timing=False)
  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  print(f"Saved {filename}")

  return 0

names = ['email-Enron','email-EU',
         'contact-high-school','contact-primary-school','hospital-lyon','malawi-village',
         'science-gallery','SFHH-conference','hypertext-conference','InVS13','InVS15',
         'tags-ask-ubuntu','tags-math-sx',
         'coauth-MAG-history', 'coauth-MAG-geology','threads-ask-ubuntu','threads-math-sx'] #'congress-bills','coauth-dblp',

steps = list(range(1,101))

params = [names, steps]
params = [p for p in product(*params)]

num_cores = 8
results = Parallel(n_jobs=num_cores)(delayed(PC_RA)(param) for param in tqdm(params))

100%|██████████| 1700/1700 [05:49<00:00,  4.87it/s]


In [ ]:
def check_if_simplex(Subsets, Supersets, IsSimplex, SimplexCount, e):
  """
  Recursive function
  """
  # By default increase simplex count by 1.
  IsSimplex[e] = True
  SimplexCount += 1;

  # Iterate over the subsets of e of size |e|-1. e is downward closed
  # iff its |e|-1 subsets belong to the hypergraph, and are all downward
  # closed themselves.
  for face in combinations(e, len(e)-1):
    if (frozenset(face) not in Subsets[e]) or (not IsSimplex[frozenset(face)]):
      # Since a subset is not a simplex, e is not a simplex
      IsSimplex[e] = False; SimplexCount -= 1
      return SimplexCount

  # If e is a simplex we now need to check if adding e made any of its |e|+1
  # size subsets a simplex, and if so, recursively check +1 subsets for closure
  for superset in Supersets[e]:
    if not IsSimplex[superset]:
      SimplexCount = check_if_simplex(Subsets, Supersets, IsSimplex, SimplexCount, superset)

  return SimplexCount

def Get_Simpliciality_SF_TS_step(H, t, SimplexCount, CandidateCount, subsets, supersets, Subsets, Supersets, IsSimplex, E, minDim = 2, maxDim = np.inf):
  """

  """

  e = H[t]

  # If edge is below or above min/max simplex size, disregard
  if (len(e) < minDim) or (len(e) > maxDim) or (e in E):
    return SimplexCount, CandidateCount
  # If edge e is min simplex size, it is a simplex but dont count it towards SF.
  # For each |e|+1 size superset of e, check if adding e made the superset a simplex
  elif len(e) == minDim:
    IsSimplex[e] = True
    supersets = {H[superset_idx] for superset_idx in supersets if len(H[superset_idx]) == len(e) + 1}
    Supersets[e].update(supersets)
    for superset in supersets:
      Subsets[superset].add(e)
      # If a superset isn't already a simplex, check if adding e made it a simplex
      if not IsSimplex[superset]:
        SimplexCount = check_if_simplex(Subsets, Supersets, IsSimplex, SimplexCount, superset)
  # Otherwise we need to check if e is a simplex
  else:
    CandidateCount += 1
    subsets = {H[subset_idx] for subset_idx in subsets if (len(H[subset_idx]) == len(e) - 1)}
    Subsets[e].update(subsets)
    for subset in subsets:
      Supersets[subset].add(e)
    supersets = {H[superset_idx] for superset_idx in supersets if len(H[superset_idx]) == len(e) + 1}
    Supersets[e].update(supersets)
    for superset in supersets:
      Subsets[superset].add(e)
    # Check if e is a simplex, and recursively check if supersets are simplices as well
    SimplexCount = check_if_simplex(Subsets, Supersets, IsSimplex, SimplexCount, e)

  E.add(e)

  return SimplexCount, CandidateCount

def Get_Simpliciality_FES_TS_step(H, t, subsets, supersets, AllSubsets, IsMaximal, minDim = 2, maxDim = np.inf):
  """

  """

  e = H[t]

  # If edge is below or above min/max simplex size, disregard
  if (len(e) < minDim) or (len(e) > maxDim):
    pass
  # If edge is min simplex size we dont include as a face in FES computation
  # but still add to subsets for any present supersets
  elif len(e) == minDim:
    for superset_idx in supersets:
      AllSubsets[H[superset_idx]].add(e)
  # Check if the newly added edge is maximal, if so add it to IsMaximal
  else:
    if len(supersets) == 0:
      IsMaximal.add(e)
      AllSubsets[e].update( {H[subset_idx] for subset_idx in subsets} )
      # All subsets of a maximal edge are not maximal
      for subset in AllSubsets[e]:
        if subset in IsMaximal:
          IsMaximal.remove(subset)
    # If not maximal, then add e to subsets of its supersets (which exist since it is not maximal)
    else:
      for superset_idx in supersets:
        AllSubsets[H[superset_idx]].add(e)

  FES = 0
  for face in IsMaximal:
    num_disregard = 0
    for i in range(1,minDim):
      num_disregard += math.comb(len(face),i)
    FES += (len(AllSubsets[face]) + 1) / ( 2 ** (len(face)) - 1 - num_disregard )

  return ( FES / len(IsMaximal) if len(IsMaximal) > 0 else 0 )

from collections import defaultdict
from array import array

class TrieNode:
    """Node in a trie for storing hyperedges."""
    __slots__ = ('children', 'timestep')  # Reduce per-instance memory overhead

    def __init__(self):
        self.children = {}
        self.timestep = None  # None if not a complete edge, timestep if complete

class HyperedgeTrie:
    """Trie structure for efficiently checking subset relationships."""
    def __init__(self):
        self.root = TrieNode()

    def insert(self, edge, timestep):
        """Insert an edge (as sorted list of nodes) into the trie."""
        sorted_edge = sorted(edge)  # Sort once
        node = self.root
        for v in sorted_edge:
            if v not in node.children:
                node.children[v] = TrieNode()
            node = node.children[v]
        node.timestep = timestep

    def find_subsets(self, sorted_edge):
        """Find all edges in the trie that are subsets of the given edge."""
        subsets = []

        def dfs(node, idx):
            # If this node represents a complete edge, it's a subset
            if node.timestep is not None:
                subsets.append(node.timestep)

            # Try to extend with remaining nodes from edge
            for i in range(idx, len(sorted_edge)):
                v = sorted_edge[i]
                if v in node.children:
                    dfs(node.children[v], i + 1)

        dfs(self.root, 0)
        return subsets


class HypergraphProcessor:
    """
    Process hypergraph using trie for subset checks and node indexing for superset checks.
    Memory-optimized version with __slots__ and compact arrays.
    """
    def __init__(self):
        self.trie = HyperedgeTrie()
        self.edges_by_timestep = {}  # timestep -> frozenset of nodes
        # Use array.array for compact integer storage instead of lists
        self.node_to_edges = defaultdict(lambda: set()) #defaultdict(lambda: array('I'))  # 'I' = unsigned int

    def add_edge_and_check(self, timestep, edge):
        """
        Add edge and check for subset/superset relationships.

        Args:
            timestep: integer timestep
            edge: set/list of integer node IDs

        Returns:
            dict with timestep, edge, is_superset_of, is_subset_of
        """
        edge_set = frozenset(edge)
        sorted_edge = sorted(edge_set)  # Sort once for reuse

        # Check for subsets using trie
        subsets_of = self.trie.find_subsets(sorted_edge)

        # Check for supersets using node indexing
        supersets_of = self._find_supersets(edge_set)

        # Add edge to trie
        self.trie.insert(sorted_edge, timestep)

        # Track which edges contain each node (using compact arrays)
        for u in edge_set:
            self.node_to_edges[u].add(timestep)##append(timestep)

        # Store the edge
        self.edges_by_timestep[timestep] = edge_set

        return timestep, sorted_edge, subsets_of, supersets_of

    def _find_supersets(self, edge):
        """
        Find all previous edges that are supersets of the given edge.
        Uses node indexing to find candidate edges efficiently.
        """
        if len(edge) == 0:
            return []

        # Find candidate edges: those that contain ALL nodes from edge
        edge_list = list(edge)

        # Start with edges containing the first node
        # Convert array to set for efficient intersection
        candidate_timesteps = self.node_to_edges[edge_list[0]]#set(self.node_to_edges[edge_list[0]])

        # Intersect with edges containing each remaining node
        for v in edge_list[1:]:
            candidate_timesteps &= self.node_to_edges[v]#set(self.node_to_edges[v])

        # Filter candidates to only those that are proper supersets
        supersets = []
        for t in candidate_timesteps:
            candidate_edge = self.edges_by_timestep[t]
            if len(candidate_edge) > len(edge):
                supersets.append(t)

        return supersets

def simpliciality_process_hypergraph(H):
    """
    Process a hypergraph dictionary incrementally.

    Args:
        H: dict mapping timestep -> set of nodes

    Returns:
        List of results for each timestep
    """
    processor = HypergraphProcessor()

    # If tqdm is available, use it; otherwise fall back to regular iteration
    try:
        from tqdm import tqdm
        iterator = tqdm(sorted(H.keys()))
    except ImportError:
        iterator = sorted(H.keys())

    upper_pairs = []
    lower_pairs = []

    # Initialize dict that keeps track of edges in H, as well as whether
    # or not they are downward closed (True or False).
    # Convert hyperedges to frozensets so they can be used as dictionary keys
    H_frozen = {t: frozenset(edge) for t, edge in H.items()}
    IsSimplex = {e: False for e in H_frozen.values()}
    Subsets = {e: set() for e in H_frozen.values()}
    Supersets = {e: set() for e in H_frozen.values()}
    SF_ts = []
    E = set()

    # Keep track of nodes which are downward closed and total number of
    # hyperedges with |e| > minDim (respectively)
    SimplexCount = 0
    CandidateCount = 0

    # Keep a set of all maximal edges, and for each edge all subsets of the edge
    # added so far.
    IsMaximal = set()
    AllSubsets = {e: set() for e in H_frozen.values()}
    FES_ts = []

    for t in iterator:
        timestep, sorted_edge, subsets, supersets = processor.add_edge_and_check(t, H[t])
        upper_pairs.append(len(subsets))
        lower_pairs.append(len(supersets))

        SimplexCount, CandidateCount = Get_Simpliciality_SF_TS_step(H_frozen, t, SimplexCount, CandidateCount, subsets, supersets, Subsets, Supersets, IsSimplex, E)
        SF_ts.append( SimplexCount / CandidateCount if CandidateCount > 0 else 0 )

        fes = Get_Simpliciality_FES_TS_step(H_frozen, t, subsets, supersets, AllSubsets, IsMaximal)
        FES_ts.append(fes)

    return (np.array(upper_pairs), np.array(lower_pairs), np.array(SF_ts), np.array(FES_ts))